In [1]:
from SteerEnergyStorage.Formulations.ElectrodeFormulations import ElectrodeFormulation
from SteerEnergyStorage.Constructions.Electrodes import Cathode, Anode
from SteerEnergyStorage.Formulations.ElectrodeAssemblies import Stack
from SteerEnergyStorage.Constructions.Cells import StackedPrismaticCell
from SteerEnergyStorage.Materials.ElectrodeMaterials import CathodeMaterial, AnodeMaterial, Binder, ConductiveAdditive
from SteerEnergyStorage.Materials.CurrentCollectors import CurrentCollector
from SteerEnergyStorage.Materials.Separators import Separator
from SteerEnergyStorage.Materials.Electrolytes import Electrolyte
from SteerEnergyStorage.Constructions.Containers import PrismaticCase, PrismaticLid, PrismaticShell

import pandas as pd
import plotly.express as px
import numpy as np

In [2]:
n = 10000

In [3]:
# construct cathodes

cathodes = []

for i in range(0, n):

    # create formulation materials
    cathode_active_material = CathodeMaterial(name=f"Faradion_Gen2_4.25V", specific_cost=11.26, density=4)
    cathode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9, name=f"SuperC65_cathode_{i}")
    cathode_binder = Binder(specific_cost=15, density = 1.7)

    # create formulation with noisy fractions
    active_material_fraction = np.random.normal(89, 1)
    binder_fraction = np.random.normal(5, 1)
    conductive_additive_fraction = 100 - active_material_fraction - binder_fraction

    cathode_formulation = ElectrodeFormulation(active_materials={cathode_active_material: active_material_fraction},
                                               binders={cathode_binder: binder_fraction},
                                               conductive_additives={cathode_conductive_additive: conductive_additive_fraction}
                                               )

    # create current collector
    cathode_current_collector = CurrentCollector(formula="Al", thickness=15, length=15, width=11.8, bare_area=16)

    # create cathode
    cathode = Cathode(formulation=cathode_formulation,
                      mass_loading=np.random.normal(10.38, 0.1),
                      current_collector=cathode_current_collector,
                      calender_density=2.6
                      )
    
    # append cathode to list
    cathodes.append(cathode) 


In [4]:
# construct anode

anodes = []

for i in range(0, n):
    # create formulation materials
    anode_active_material = AnodeMaterial(name="faradion_hc", specific_cost=14.27, density=1.5)
    anode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9)
    anode_binder = Binder(specific_cost=10, density=1.7)

    # create formulation with noisy fractions
    active_material_fraction = np.random.normal(88, 1)
    binder_fraction = np.random.normal(3, 1)
    conductive_additive_fraction = 100 - active_material_fraction - binder_fraction

    # create formulation
    anode_formulation = ElectrodeFormulation(active_materials={anode_active_material: active_material_fraction},
                                            binders={anode_binder: binder_fraction},
                                            conductive_additives={anode_conductive_additive: conductive_additive_fraction})

    # create current collector
    anode_current_collector = CurrentCollector(formula="Cu", thickness=4, length=15, width=11.8, bare_area=7.55)

    # create anode
    anode = Anode(formulation=anode_formulation,
                  mass_loading=np.random.normal(5.20, 0.1),
                  current_collector=anode_current_collector,
                  calender_density=0.85)
    
    anodes.append(anode)


In [5]:
# Other cell components

seperators = []
stacks = []
for i in range(0, n):
    # construct seperator
    seperator = Separator(thickness=16, areal_cost=0.9, density=0.4, width=12, porosity=47, fold_length=18.6)
    seperators.append(seperator)


In [6]:

# make electrolyte
electrolyte = Electrolyte(name="LiPF6", specific_cost=8.94, density=1.2)

cases = []
stacks = []
for i in range(0, n):
    prismatic_lid = PrismaticLid(cost=0.05, mass=10, external_width=1.3, internal_width=0.8)
    prismatic_shell = PrismaticShell(cost=0.15, mass=60, internal_length=19, internal_width=13, internal_height=0.5, wall_thickness=0.5)
    prismatic_case = PrismaticCase(lid=prismatic_lid, shell=prismatic_shell)
    cases.append(prismatic_case)

    stack = prismatic_case.get_optimized_stack(cathode=cathodes[i], anode=anodes[i], separator=seperators[i])
    stacks.append(stack)


In [7]:
cells = []
for i in range(0, n):
    cell = StackedPrismaticCell(stack=stacks[i],
                                prismatic_case=cases[i],
                                electrolyte=electrolyte,
                                electrolyte_overfill=10,
                                reversible_capacity=10,
                                irreversible_capacity=1)
    
    cells.append(cell)

In [8]:
cells[0].get_capacity_voltage_plot(width=900, height=500)

In [9]:
figure = cells[0].get_mass_breakdown_plot(width=900, height=400)
figure.show()

In [10]:
figure = cells[0].get_cost_breakdown_plot(width=900, height=400)
figure.show()

In [11]:
# look at the distribution of the cost of the cell stacks

plot_data = pd.DataFrame({
    'cell_name': [c.name for c in cells],
    'energy': [c.energy for c in cells]
})

px.histogram(plot_data, x='energy', nbins=100, template='presentation', 
             labels={'energy': 'energy / kWh'}, width=700, height=400).show()

In [12]:
# look at the distribution of the cost of the cell stacks

plot_data = pd.DataFrame({
    'cell_name': [c.name for c in cells],
    'stack_cost': [c.cost_breakdown[s] for c, s in zip(cells, stacks)]
})

px.histogram(plot_data, x='stack_cost', nbins=100, template='presentation', 
             labels={'stack_cost': 'Stack cost $'}, width=700, height=400).show()

KeyError: stack

In [ ]:
# look at the distribution of the masses of the cells

plot_data = pd.DataFrame({
    'cell_name': [c.name for c in cells],
    'cell_mass': [c.mass for c in cells]
})

px.histogram(plot_data, x='cell_mass', title='Distribution of cell masses', nbins=100, template='presentation', 
             labels={'cell_mass': 'Mass [g]'}, width=700, height=400).show()